# Introduction

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

# Data Loading

We load the CSV file using a more explicit configuration to avoid warnings.  
If we used the default settings, Pandas would raise a `DtypeWarning` because it would not correctly infer the date column while reading the file in chunks.  

By disabling chunking (`low_memory=False`) and explicitly parsing the date column (`parse_dates=[7]`), the import becomes slightly slower but much more accurate and stable.


In [2]:
data = pd.read_csv(
    '../Datasets/UKDataset/UKdata.csv',
    parse_dates=[7],
    dayfirst=False,
    low_memory=False
)

data["Death_date"] = pd.to_datetime(data["Death_date"], errors="coerce")
data["Vdate_0"]    = pd.to_datetime(data["Vdate_0"], errors="coerce")

In [3]:
#To display all columns and all rows when printing a dataframe
pd.options.display.max_columns = None
pd.options.display.max_rows = None

# Data Understanding

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502412 entries, 0 to 502411
Columns: 293 entries, Sex to Familiarity_siblings__Prostate_cancer__0
dtypes: datetime64[ns](3), float64(165), int64(124), object(1)
memory usage: 1.1+ GB


In [5]:
data.dtypes.value_counts()

float64           165
int64             124
datetime64[ns]      3
object              1
Name: count, dtype: int64

## Exploring outcome-related columns

We now explore the outcome-related columns in the dataset, which we will later use to define our prediction labels. Obviously, we will remove these columns from the dataset once the labels have been defined.

Below we provide a brief description of the outcome-related columns we will use to define our prediction labels:

- `Follow_up_time`  
  Represents the follow-up time in **days**. This is the primary column to use when evaluating survival status for participants who **did not die**, since it tells us how long they were observed. If a participant is alive, this is the time window we compare to our survival threshold (e.g., 10 years).

- `Time_to_death`  
  Number of days between the baseline visit and the date of death. It is only defined for individuals who actually died.

- `Lost_FU`  
  The date at which the participant was **lost to follow-up**, meaning they stopped providing information before the end of the study.  
  This can be confused with *Follow_up_time*, but the concepts are different:  
  - *Follow_up_time* is a duration (in days).  
  - *Lost_FU* is a calendar date.  
  
  Participants with a non-null *Lost_FU* left the study early, so their survival time must be interpreted with caution.  
  When both pieces of information are available, **Follow_up_time should be considered the primary and more reliable indicator of follow-up duration**, as it directly expresses the number of observed days and is consistent with how survival thresholds (e.g., 10 years) are evaluated.


- `Death_date`  
  The calendar date of death. 

- `Cause_of_death`  
  Contains the cause of death encoded using **ICD-10 codes**. Since the raw codes are too granular for our analysis, we apply a mapping function that groups them into broader **macro-categories** (e.g., “Cancer”, “Cardiovascular disease”, “Respiratory disease”, etc.).

- `Cause_group`  
  The macro-category derived from the ICD-10 code. This grouped version is more interpretable and more suitable for downstream modeling.

- `Vdate_0`  
  Date of the participant’s **baseline assessment**. 

- `Vmonth_0`  
  The month (as an integer) of the baseline assessment.

- `Elapsed_time_visit0_death0`  
  Derived number of days between the baseline visit and the death date (when applicable). Useful as a consistency check with *Time_to_death*.

- `Age_Death`  
  Age (in years) at the moment of death. Missing for participants who are still alive.

- `label_10`  
  The original label included in the dataset. It indicates whether a participant **died within 10 years**, without distinguishing the cause of death.  
  This label will serve as a reference, but additional outcome definitions may be introduced depending on the study goals.


In [6]:
cols = [
    "Follow_up_time", "Time_to_death", "Lost_FU", "Death_date",
    "Cause_of_death", "Vdate_0", "Vmonth_0", "Elapsed_time_visit0_death0",
    "Age_Death", "label_10"
]


In [11]:
def icd_group(code):
    """
    Map an ICD-10 cause-of-death code to a broader macro-category.

    Parameters
    ----------
    code : str or NaN
        ICD-10 code describing the specific cause of death.

    Returns
    -------
    str or NaN
        A macro-category describing the general type of cause:
        - "Cancer" for codes starting with 'C'
        - "Cardiovascular disease" for codes starting with 'I'
        - "Respiratory disease" for codes starting with 'J'
        - "Mental/neurological" for codes starting with 'F'
        - "Nervous system" for codes starting with 'G'
        - "COVID-19 / special codes" for codes starting with 'U'
        - "Other" for any other ICD-10 group

    Notes
    -----
    The function uses only the first character of the ICD-10 code,
    since this character indicates the ICD-10 chapter and therefore
    the high-level disease category.
    """

    # If the value is missing, return NaN
    if pd.isna(code):
        return np.nan

    # Extract the first character (ICD-10 chapter)
    c = str(code)[0]

    # Map ICD-10 chapters to macro-categories
    if c == "C":
        return "Cancer"
    if c == "I":
        return "Cardiovascular disease"
    if c == "J":
        return "Respiratory disease"
    if c == "F":
        return "Mental/neurological"
    if c == "G":
        return "Nervous system"
    if c == "U":
        return "COVID-19 / special codes"

    # Default category for all other ICD-10 groups
    return "Other"


In [12]:
data["Cause_group"] = data["Cause_of_death"].apply(icd_group)

In [13]:
cols=cols + ["Cause_group"]

In [31]:
data.label_10.value_counts()

label_10
1.0    427406
0.0     34940
Name: count, dtype: int64

In [34]:
data.loc[data["Follow_up_time"] != 0, "label_10"].value_counts()


label_10
1.0    20614
0.0     1132
Name: count, dtype: int64

In [14]:
data.label_10.isna().sum()

40066

In [58]:
data.Cause_group.value_counts()

Cause_group
Cancer                      17595
Cardiovascular disease       7130
Other                        4003
Respiratory disease          2489
Nervous system               1884
COVID-19 / special codes     1095
Mental/neurological           719
Name: count, dtype: int64

# Data Cleaning

## New Labels Creation

## PCA?

## Qualche plot (magari sui label)

## Esportare i dataset